# Prophet Evaluation and Comparison

Prophet includes built-in tools for time-series cross-validation and
performance evaluation.  This notebook covers:

1. Prophet's `cross_validation()` — simulated historical forecasts
2. `performance_metrics()` — MAE, RMSE, MAPE, coverage
3. Visualizing cross-validation metrics over forecast horizon
4. Head-to-head comparison: Prophet vs SARIMA
5. When to use Prophet vs classical methods
6. Limitations of Prophet

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load and prepare data
airline = pd.read_csv(DATA_DIR / "airline_passengers.csv")
airline_prophet = airline.rename(columns={
    "Month": "ds",
    "Thousands of Passengers": "y",
})
airline_prophet["ds"] = pd.to_datetime(airline_prophet["ds"])

alcohol = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv")
alcohol_prophet = alcohol.rename(columns={
    "DATE": "ds",
    "S4248SM144NCEN": "y",
})
alcohol_prophet["ds"] = pd.to_datetime(alcohol_prophet["ds"])

print(f"Airline passengers: {len(airline_prophet)} months")
print(f"Alcohol sales:      {len(alcohol_prophet)} months")

---
## 1. Prophet's Built-in Cross-Validation

Prophet provides `cross_validation()` which performs **simulated historical
forecasts** (also called time-series cross-validation or backtesting).

The idea: train on data up to a cutoff date, forecast `horizon` steps
ahead, then slide the cutoff forward by `period` and repeat.

```
|--- initial ---|--- horizon ---|
                |
                cutoff_1
                     |--- period ---|--- horizon ---|
                                    |
                                    cutoff_2
```

### Parameters

| Parameter | Meaning | Example |
|-----------|---------|---------|
| `initial` | Minimum training period | `'730 days'` (2 years) |
| `period` | Step size between cutoffs | `'180 days'` (6 months) |
| `horizon` | Forecast horizon to evaluate | `'365 days'` (1 year) |

In [ ]:
# Fit a model on the full alcohol sales data for cross-validation
model_alc = Prophet(seasonality_mode="multiplicative")
model_alc.fit(alcohol_prophet)

print("Model fitted on full alcohol sales data.")
print("Now running cross-validation — this may take a minute...")

In [ ]:
# Run cross-validation
# initial: first 10 years of training
# period: slide cutoff by 180 days (6 months)
# horizon: forecast 365 days (1 year) ahead
cv_results = cross_validation(
    model_alc,
    initial="3650 days",   # ~10 years of initial training
    period="180 days",     # slide cutoff every 6 months
    horizon="365 days",    # forecast 1 year ahead
)

print(f"Cross-validation results shape: {cv_results.shape}")
print(f"Number of cutoff dates: {cv_results['cutoff'].nunique()}")
print()
cv_results.head(10)

In [ ]:
# Examine the cross-validation output columns
print("Cross-validation DataFrame columns:")
print(f"  ds:         forecast date")
print(f"  yhat:       predicted value")
print(f"  yhat_lower: lower uncertainty bound")
print(f"  yhat_upper: upper uncertainty bound")
print(f"  y:          actual value")
print(f"  cutoff:     training cutoff date for this fold")
print()
print(f"Cutoff dates:")
for i, cutoff in enumerate(sorted(cv_results["cutoff"].unique())):
    print(f"  Fold {i+1}: cutoff = {pd.Timestamp(cutoff).date()}")

---
## 2. Performance Metrics

Prophet's `performance_metrics()` computes several error metrics as a
function of the forecast horizon:

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **MAE** | $\frac{1}{n}\sum\|y - \hat{y}\|$ | Average absolute error (same units as data) |
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | Average squared error |
| **RMSE** | $\sqrt{\text{MSE}}$ | Root mean squared error (same units as data) |
| **MAPE** | $\frac{1}{n}\sum\left\|\frac{y - \hat{y}}{y}\right\| \times 100$ | Percentage error |
| **MDAPE** | median of $\left\|\frac{y - \hat{y}}{y}\right\| \times 100$ | Median percentage error (robust to outliers) |
| **Coverage** | fraction of $y$ within $[\hat{y}_\text{lower}, \hat{y}_\text{upper}]$ | How often actuals fall inside the uncertainty interval |

In [ ]:
# Compute performance metrics
metrics = performance_metrics(cv_results)

print(f"Metrics shape: {metrics.shape}")
print()
print("Metrics are computed in rolling windows over the forecast horizon.")
print("Each row corresponds to a different horizon window.\n")
metrics.head(10)

In [ ]:
# Overall summary
print("Alcohol Sales — Cross-Validation Summary (Multiplicative Prophet)")
print("=" * 55)
print(f"  MAE:      {metrics['mae'].mean():.2f}")
print(f"  RMSE:     {metrics['rmse'].mean():.2f}")
print(f"  MAPE:     {metrics['mape'].mean() * 100:.2f}%")
print(f"  MDAPE:    {metrics['mdape'].mean() * 100:.2f}%")
print(f"  Coverage: {metrics['coverage'].mean() * 100:.1f}%")
print()
print("Coverage close to 80% means the 80% uncertainty intervals")
print("are well-calibrated.")

---
## 3. Visualizing Cross-Validation Metrics

Prophet provides `plot_cross_validation_metric()` to show how each
metric changes as the forecast horizon increases.  Metrics should
generally get worse at longer horizons.

In [ ]:
# Plot MAPE over forecast horizon
fig = plot_cross_validation_metric(cv_results, metric="mape")
plt.title("MAPE vs Forecast Horizon — Alcohol Sales")
plt.tight_layout()
plt.show()

print("Blue dots = MAPE for individual forecast points.")
print("Blue line = rolling mean.  MAPE should increase with horizon.")

In [ ]:
# Plot RMSE and Coverage side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_cross_validation_metric(cv_results, metric="rmse", ax=axes[0])
axes[0].set_title("RMSE vs Forecast Horizon")

plot_cross_validation_metric(cv_results, metric="coverage", ax=axes[1])
axes[1].set_title("Coverage vs Forecast Horizon")
axes[1].axhline(0.8, color="red", linestyle="--", alpha=0.5, label="80% target")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Coverage near 0.8 means the 80% prediction intervals are well-calibrated.")

In [ ]:
# Visualize the cross-validation folds
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(alcohol_prophet["ds"], alcohol_prophet["y"], color="black", alpha=0.3, label="Full data")

colors = plt.cm.tab10(np.linspace(0, 1, cv_results["cutoff"].nunique()))
for i, (cutoff, group) in enumerate(cv_results.groupby("cutoff")):
    cutoff_date = pd.Timestamp(cutoff)
    ax.axvline(cutoff_date, color=colors[i], alpha=0.3, linestyle="--")
    ax.plot(group["ds"], group["yhat"], color=colors[i], alpha=0.6, linewidth=1)

ax.set_title("Cross-Validation Folds — Each Color is a Different Cutoff")
ax.set_xlabel("Date")
ax.set_ylabel("Millions of Dollars")
plt.tight_layout()
plt.show()

print("Each colored line is a forecast from a different cutoff date.")
print("Dashed vertical lines mark the cutoff dates.")

---
## 4. Cross-Validation on Airline Passengers

Let us also cross-validate the airline passengers model to see how
Prophet performs on a shorter series.

In [ ]:
# Fit multiplicative Prophet on full airline data
model_air = Prophet(seasonality_mode="multiplicative")
model_air.fit(airline_prophet)

# Cross-validation with shorter periods (smaller dataset)
cv_air = cross_validation(
    model_air,
    initial="1460 days",   # ~4 years initial training
    period="180 days",     # slide every 6 months
    horizon="365 days",    # forecast 1 year ahead
)

metrics_air = performance_metrics(cv_air)

print("Airline Passengers — Cross-Validation Summary (Multiplicative Prophet)")
print("=" * 60)
print(f"  MAE:      {metrics_air['mae'].mean():.2f}")
print(f"  RMSE:     {metrics_air['rmse'].mean():.2f}")
print(f"  MAPE:     {metrics_air['mape'].mean() * 100:.2f}%")
print(f"  Coverage: {metrics_air['coverage'].mean() * 100:.1f}%")

In [ ]:
# Plot MAPE for airline passengers
fig = plot_cross_validation_metric(cv_air, metric="mape")
plt.title("MAPE vs Forecast Horizon — Airline Passengers")
plt.tight_layout()
plt.show()

---
## 5. Head-to-Head: Prophet vs SARIMA

The classic comparison.  We will fit both Prophet and SARIMA on the
airline passengers data and compare their forecasts on a held-out test set.

This is a fair test because:
- Both models see the same training data
- Both are evaluated on the same test period
- We use a tuned Prophet (multiplicative) and auto-selected SARIMA

In [ ]:
# Prepare train/test for both models
n_test = 24
train_air = airline_prophet.iloc[:-n_test].copy()
test_air = airline_prophet.iloc[-n_test:].copy()

# Also prepare a pandas Series version for statsmodels
airline_ts = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline_ts.index.freq = "MS"
airline_ts.columns = ["Passengers"]

train_ts = airline_ts.iloc[:-n_test]
test_ts = airline_ts.iloc[-n_test:]

print(f"Train: {len(train_air)} months")
print(f"Test:  {len(test_air)} months")

In [ ]:
# Fit Prophet (multiplicative)
model_prophet = Prophet(seasonality_mode="multiplicative")
model_prophet.fit(train_air)
future = model_prophet.make_future_dataframe(periods=n_test, freq="ME")
forecast_prophet = model_prophet.predict(future)
fc_prophet_test = forecast_prophet.set_index("ds").loc[test_air["ds"].values].reset_index()

prophet_pred = fc_prophet_test["yhat"].values
actual = test_air["y"].values

print("Prophet (multiplicative) fitted.")

In [ ]:
# Fit SARIMA using statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

# SARIMA(0,1,1)(0,1,1)[12] — the classic Box-Jenkins airline model
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sarima = SARIMAX(
        train_ts["Passengers"],
        order=(0, 1, 1),
        seasonal_order=(0, 1, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarima_result = sarima.fit(disp=False)

sarima_forecast = sarima_result.forecast(steps=n_test)
sarima_pred = sarima_forecast.values

print("SARIMA(0,1,1)(0,1,1)[12] fitted.")
print(f"AIC: {sarima_result.aic:.2f}")

In [ ]:
# Compare metrics
print("=" * 65)
print("HEAD-TO-HEAD: Airline Passengers (24-month holdout)")
print("=" * 65)
print(f"{'Model':<30s} {'MAE':>8s} {'RMSE':>8s} {'MAPE':>8s}")
print("-" * 58)

for name, pred in [("Prophet (multiplicative)", prophet_pred),
                   ("SARIMA(0,1,1)(0,1,1)[12]", sarima_pred)]:
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f"{name:<30s} {mae:8.2f} {rmse:8.2f} {mape:7.2f}%")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(train_air["ds"], train_air["y"], color="tab:blue", alpha=0.5, label="Train")
ax.plot(test_air["ds"], test_air["y"], color="black",
        marker="o", markersize=5, linewidth=2, label="Actual")

ax.plot(test_air["ds"], prophet_pred,
        color="tab:red", linestyle="--", linewidth=2, label="Prophet")
ax.plot(test_air["ds"], sarima_pred,
        color="tab:green", linestyle="--", linewidth=2, label="SARIMA")

ax.set_title("Airline Passengers — Prophet vs SARIMA")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Month-by-month error comparison
errors = pd.DataFrame({
    "Month": test_air["ds"].values,
    "Actual": actual,
    "Prophet": prophet_pred,
    "SARIMA": sarima_pred,
})
errors["Prophet_Error"] = errors["Actual"] - errors["Prophet"]
errors["SARIMA_Error"] = errors["Actual"] - errors["SARIMA"]
errors["Prophet_APE"] = np.abs(errors["Prophet_Error"] / errors["Actual"]) * 100
errors["SARIMA_APE"] = np.abs(errors["SARIMA_Error"] / errors["Actual"]) * 100

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(errors))
width = 0.35
ax.bar(x - width/2, errors["Prophet_APE"], width, label="Prophet APE%", color="tab:red", alpha=0.7)
ax.bar(x + width/2, errors["SARIMA_APE"], width, label="SARIMA APE%", color="tab:green", alpha=0.7)
ax.set_xticks(x[::3])
ax.set_xticklabels([pd.Timestamp(d).strftime("%Y-%m") for d in errors["Month"].values[::3]], rotation=45)
ax.set_ylabel("Absolute Percentage Error (%)")
ax.set_title("Month-by-Month Forecast Error: Prophet vs SARIMA")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. When to Use Prophet vs ARIMA/ETS

Neither method is universally better.  The right choice depends on
the data characteristics and the use case.

### Prophet works best when:
- Data is **daily or weekly** with multiple seasonalities (yearly + weekly + daily)
- There are **holiday effects** or known events
- The data has **missing values** or **outliers** (Prophet handles these gracefully)
- You need **fast, automated** forecasting for many series (business dashboards)
- **Interpretability** of components (trend, seasonality) is important
- The series is long (2+ years of daily data is ideal)

### ARIMA/ETS works best when:
- The series is **short** (< 2 years, < 100 observations)
- You need **narrow prediction intervals** with statistical guarantees
- The data follows a well-understood **ARIMA process** (strong autocorrelation)
- You need **statistical rigor** (hypothesis tests, model diagnostics)
- The data is **monthly or quarterly** with a single seasonal period
- You are working in a domain where Box-Jenkins methodology is standard

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    "Feature": [
        "Best data frequency",
        "Handles multiple seasonalities",
        "Handles missing data",
        "Holiday effects",
        "Prediction intervals",
        "Short series (<100 obs)",
        "Requires differencing",
        "Interpretable components",
        "Statistical tests (ADF, Ljung-Box)",
        "Automated fitting",
        "Multivariate support",
    ],
    "Prophet": [
        "Daily/Weekly",
        "Yes (yearly+weekly+daily+custom)",
        "Yes (ignores gaps)",
        "Yes (built-in)",
        "Wide, simulation-based",
        "Poor",
        "No (handles trend internally)",
        "Yes (trend, seasonality, holidays)",
        "No (uses cross-validation instead)",
        "Yes (minimal tuning needed)",
        "No (univariate only)",
    ],
    "ARIMA/ETS": [
        "Monthly/Quarterly",
        "One seasonal period",
        "Requires complete data",
        "Via regressors (manual)",
        "Narrower, model-based",
        "Good",
        "Yes (d and D parameters)",
        "Partial (ETS components)",
        "Yes (full diagnostic suite)",
        "Via auto_arima / ETS",
        "VAR for multivariate",
    ],
})

print(comparison.to_string(index=False))

---
## 7. Limitations of Prophet

Prophet is a powerful tool, but it is not a silver bullet.  Understanding
its limitations helps you know when to reach for something else.

In [ ]:
# Limitation 1: Short series — Prophet needs sufficient data to learn
# seasonal patterns. Let us try it on just 2 years of airline data.
short_air = airline_prophet.iloc[:24].copy()  # only 2 years

model_short = Prophet(seasonality_mode="multiplicative")
model_short.fit(short_air)
future_short = model_short.make_future_dataframe(periods=12, freq="ME")
forecast_short = model_short.predict(future_short)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(short_air["ds"], short_air["y"], "ko-", markersize=4, label="Training data (2 years)")
ax.plot(forecast_short["ds"].iloc[24:], forecast_short["yhat"].iloc[24:],
        "r--", label="Prophet forecast")
ax.fill_between(
    forecast_short["ds"].iloc[24:],
    forecast_short["yhat_lower"].iloc[24:],
    forecast_short["yhat_upper"].iloc[24:],
    alpha=0.2, color="red",
)
# Plot what actually happened
ax.plot(airline_prophet["ds"].iloc[24:36], airline_prophet["y"].iloc[24:36],
        "g-o", markersize=4, label="Actual (next year)")
ax.set_title("Limitation: Prophet on a Short Series (only 2 years)")
ax.legend()
plt.tight_layout()
plt.show()

print("With only 2 years of data, Prophet has limited ability to learn")
print("the seasonal pattern and produces wide uncertainty intervals.")

In [ ]:
# Limitation 2: Wide uncertainty intervals at long horizons
model_long = Prophet(seasonality_mode="multiplicative")
model_long.fit(airline_prophet)

# Forecast 5 years ahead
future_long = model_long.make_future_dataframe(periods=60, freq="ME")
forecast_long = model_long.predict(future_long)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(airline_prophet["ds"], airline_prophet["y"], "k-", alpha=0.5, label="Observed")
forecast_future = forecast_long.iloc[len(airline_prophet):]
ax.plot(forecast_future["ds"], forecast_future["yhat"], "r-", label="Forecast")
ax.fill_between(
    forecast_future["ds"],
    forecast_future["yhat_lower"],
    forecast_future["yhat_upper"],
    alpha=0.2, color="red", label="Uncertainty",
)
ax.set_title("Limitation: Uncertainty Grows Rapidly at Long Horizons")
ax.legend()
plt.tight_layout()
plt.show()

print("Prophet's uncertainty intervals widen quickly with forecast horizon.")
print("At 5 years out, the intervals may be too wide to be useful.")

### Summary of Limitations

1. **Short series**: Prophet needs sufficient history to learn seasonal
   patterns.  For series with fewer than ~50 observations, classical
   methods (ARIMA, ETS) often perform better.

2. **No multivariate support**: Prophet models each series independently.
   It cannot capture cross-series dependencies (use VAR or machine
   learning methods for that).

3. **Overfitting risk**: Adding too many custom seasonalities, holidays,
   or regressors can lead to overfitting — always validate with
   cross-validation.

4. **Wide uncertainty intervals**: Prophet's prediction intervals are
   simulation-based and can be very wide, especially at long horizons.
   ARIMA often produces tighter intervals.

5. **Not great for all frequencies**: Prophet was designed for daily/weekly
   business data.  For high-frequency data (hourly, minute-level) or
   very low-frequency data (annual), other methods may be more appropriate.

6. **Trend extrapolation**: Prophet extrapolates the trend linearly into
   the future, which can lead to unreasonable forecasts if the trend
   changes direction after the training period.

---
## Summary

- **Cross-validation**: `cross_validation(model, initial, period, horizon)` performs
  time-series backtesting by training on expanding windows and forecasting ahead

- **Performance metrics**: `performance_metrics(cv_results)` computes MAE, MSE, RMSE,
  MAPE, MDAPE, and coverage as functions of the forecast horizon

- **Visualization**: `plot_cross_validation_metric(cv_results, metric)` shows how
  error changes with forecast distance

- **Prophet vs SARIMA**: neither is universally better
  - Prophet excels at daily/weekly data with holidays and multiple seasonalities
  - SARIMA excels at short monthly/quarterly series with strong autocorrelation
  - Always compare on your specific data with proper cross-validation

- **Limitations**: Prophet struggles with short series, provides no multivariate
  support, can produce wide uncertainty intervals, and risks overfitting with
  too many regressors

In [ ]:
print("End of Chapter 11: Prophet.")
print()
print("Key import to remember: from prophet import Prophet")
print("(NOT fbprophet — the package was renamed in 2021)")